# Stage 14 check - preflight and smoke runs

Design: `docs/stage14_data_scale.md`.

## Setup

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/RAG chunk optimize'
ACCOUNT_LABEL = 'A'          # run-log identifier
CLEAR_STALE_LOCK = False     # requires a stopped lock holder

from google.colab import drive
drive.mount('/content/drive')

import os, shlex, subprocess, sys
if not os.path.isfile(os.path.join(PROJECT_DIR, 'config.py')):
    raise RuntimeError(f'no config.py under {PROJECT_DIR!r} - the shared folder is not mounted at this path.')
os.environ['RAG_DATA_ROOT'] = PROJECT_DIR + '/artifacts'
os.environ['PYTHONUNBUFFERED'] = '1'
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
GUARD = f' --account {ACCOUNT_LABEL}' + (' --clear-stale-lock' if CLEAR_STALE_LOCK else '')


def run(cmd):
    """Stream command output and raise on a nonzero exit."""
    proc = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'command failed: {cmd}')

## Install dependencies

In [ ]:
run('pip install -q -r requirements.txt')

## Preflight

In [ ]:
run('python -u scripts/35_preflight_stage14.py --gpu')

## Data smoke check

In [ ]:
run('python -u scripts/32_build_stage14_data.py --smoke')

## Training smoke check

In [ ]:
run('python -u scripts/33_train_data_scale.py --smoke' + GUARD)